# IWM Historical Stock Price Analysis
## Technical Indicators and Trading Signal Generation

This notebook provides an interactive analysis of IWM stock data with various technical indicators and put/call signal generation based on price movement patterns.

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, time
import os
import glob
from typing import Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set plot style - use available style
plt.style.use('seaborn-darkgrid' if 'seaborn-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 8)

In [3]:
# Import the analyzer class from our script
from iwm_analysis import IWMAnalyzer

# Initialize analyzer
analyzer = IWMAnalyzer()

## Step 1: Combine CSV Files

In [4]:
# Check if the combined file already exists
output_file = "/workspace/data/historical_iwm_0824_0825.csv"

if os.path.exists(output_file):
    print(f"Loading existing combined file: {output_file}")
    df = pd.read_csv(output_file)
    df['Time'] = pd.to_datetime(df['Time'])
else:
    print("Combined file not found. Please ensure historical_iwm_0824_0825.csv exists in the data folder.")
    raise FileNotFoundError(f"{output_file} not found")

# Display basic info
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nDate range: {df['Time'].min()} to {df['Time'].max()}")
print(f"\nFirst few rows:")
df.head()

Combined file not found. Please ensure historical_iwm_0824_0825.csv exists in the data folder.


FileNotFoundError: /workspace/data/historical_iwm_0824_0825.csv not found

In [ ]:
# Check data quality
print("Missing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
df.describe()

## Step 2: Calculate Technical Indicators

In [ ]:
# Check if indicators file already exists
enhanced_file = output_file.replace('.csv', '_with_indicators.csv')

if os.path.exists(enhanced_file):
    print(f"Loading existing indicators file: {enhanced_file}")
    df = pd.read_csv(enhanced_file)
    df['Time'] = pd.to_datetime(df['Time'])
else:
    print("Indicators file not found. Running iwm_analysis.py to generate indicators...")
    # Run the analysis script
    import subprocess
    result = subprocess.run(['python3', 'iwm_analysis.py'], capture_output=True, text=True)
    if result.returncode == 0:
        print("Analysis completed successfully!")
        df = pd.read_csv(enhanced_file)
        df['Time'] = pd.to_datetime(df['Time'])
    else:
        print(f"Error running analysis: {result.stderr}")
        raise RuntimeError("Failed to generate indicators")

# Display sample of data with indicators
print("Sample data with indicators:")
indicator_cols = ['Time', 'Last', 'Volume', 'ATR14_W', 'RSI14_W', 'EMA9', 'EMA20', 'EMA50', 'VWAP', 'RVOL20', 'StochRSI_K']
available_cols = [col for col in indicator_cols if col in df.columns]
df[available_cols].tail(20)

## Step 3: Visualize Price and Indicators

In [ ]:
# Create subplots for visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# Plot 1: Price with EMAs
axes[0].plot(df['Time'], df['Last'], label='Price', color='black', linewidth=1)
axes[0].plot(df['Time'], df['EMA9'], label='EMA9', color='blue', alpha=0.7)
axes[0].plot(df['Time'], df['EMA20'], label='EMA20', color='orange', alpha=0.7)
axes[0].plot(df['Time'], df['EMA50'], label='EMA50', color='red', alpha=0.7)
axes[0].plot(df['Time'], df['VWAP'], label='VWAP', color='purple', alpha=0.7, linestyle='--')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='best')
axes[0].set_title('IWM Price with Moving Averages and VWAP')
axes[0].grid(True, alpha=0.3)

# Plot 2: Volume and RVOL
axes[1].bar(df['Time'], df['Volume'], alpha=0.3, color='gray', label='Volume')
ax1_twin = axes[1].twinx()
ax1_twin.plot(df['Time'], df['RVOL20'], label='RVOL20', color='green', linewidth=2)
ax1_twin.axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Volume')
ax1_twin.set_ylabel('RVOL')
axes[1].legend(loc='upper left')
ax1_twin.legend(loc='upper right')
axes[1].set_title('Volume and Relative Volume')
axes[1].grid(True, alpha=0.3)

# Plot 3: RSI
axes[2].plot(df['Time'], df['RSI14_W'], label='RSI(14)', color='blue', linewidth=2)
axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Overbought')
axes[2].axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Oversold')
axes[2].fill_between(df['Time'], 30, 70, alpha=0.1, color='gray')
axes[2].set_ylabel('RSI')
axes[2].set_ylim(0, 100)
axes[2].legend(loc='best')
axes[2].set_title('Relative Strength Index (Wilder)')
axes[2].grid(True, alpha=0.3)

# Plot 4: Stochastic RSI
axes[3].plot(df['Time'], df['StochRSI_K'], label='StochRSI %K', color='blue', linewidth=2)
axes[3].plot(df['Time'], df['StochRSI_D'], label='StochRSI %D', color='red', linewidth=2)
axes[3].axhline(y=80, color='red', linestyle='--', alpha=0.5)
axes[3].axhline(y=20, color='green', linestyle='--', alpha=0.5)
axes[3].fill_between(df['Time'], 20, 80, alpha=0.1, color='gray')
axes[3].set_ylabel('StochRSI')
axes[3].set_ylim(0, 100)
axes[3].legend(loc='best')
axes[3].set_title('Stochastic RSI')
axes[3].set_xlabel('Time')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Generate Trading Signals

In [ ]:
# Load existing signals if available
signals_file = "/workspace/data/historical_iwm_0824_0825_signals.csv"

if os.path.exists(signals_file):
    print(f"Loading existing signals from: {signals_file}")
    signals_df = pd.read_csv(signals_file)
    
    # Display signal summary
    print(f"\nTotal signals: {len(signals_df)}")
    
    # Count by signal type
    signal_counts = signals_df['signal_type'].value_counts()
    print("\nSignals by type:")
    for signal_type, count in signal_counts.items():
        print(f"  {signal_type}: {count}")
else:
    print("No signals file found. The current analysis uses technical indicator signals.")
    print("Run iwm_analysis.py to generate signal outputs.")

In [ ]:
# Generate trading signals
signals_file = "/workspace/data/historical_iwm_0824_0825_signals.csv"
signals_df = analyzer.generate_signals(df, runs)

# Save signals
signals_df.to_csv(signals_file, index=False)
print(f"Signals saved to {signals_file}")

# Display signal summary
print(f"\nTotal signals generated: {len(signals_df)}")
print(f"Call signals: {len(signals_df[signals_df['trade_type'] == 'call'])}")
print(f"Put signals: {len(signals_df[signals_df['trade_type'] == 'put'])}")

# Display first few signals
print("\nFirst 5 signals:")
signals_df[['trade_type', 'entry_timestamp', 'exit_timestamp', 'entry_price', 'exit_price', 'return_pct']].head()

## Step 5: Analyze Signal Performance

In [ ]:
# Analyze your actual trading patterns
if os.path.exists('data/trade_patterns.csv'):
    patterns_df = pd.read_csv('data/trade_patterns.csv', index_col=0)
    print("Your Actual Trading Pattern Analysis:")
    print("="*50)
    
    # Group patterns by trade type
    call_patterns = {k: v for k, v in patterns_df.to_dict('index').items() if k.startswith('CALL')}
    put_patterns = {k: v for k, v in patterns_df.to_dict('index').items() if k.startswith('PUT')}
    
    # Display CALL patterns
    if call_patterns:
        print("\nCALL Patterns:")
        for pattern_key, data in call_patterns.items():
            exit_type = pattern_key.replace('CALL_', '')
            print(f"\n  {exit_type}:")
            print(f"    Count: {data['count']:.0f} trades")
            print(f"    Win Rate: {data['profitable_pct']:.1f}%")
            print(f"    Average Return: {data['avg_return']:.3f}%")
            print(f"    Average Duration: {data['avg_duration']:.1f} minutes")
            print(f"    Entry RSI: {data['entry_rsi_mean']:.1f} ± {data['entry_rsi_std']:.1f}")
    
    # Display PUT patterns
    if put_patterns:
        print("\n\nPUT Patterns:")
        for pattern_key, data in put_patterns.items():
            exit_type = pattern_key.replace('PUT_', '')
            print(f"\n  {exit_type}:")
            print(f"    Count: {data['count']:.0f} trades")
            print(f"    Win Rate: {data['profitable_pct']:.1f}%")
            print(f"    Average Return: {data['avg_return']:.3f}%")
            print(f"    Average Duration: {data['avg_duration']:.1f} minutes")
            print(f"    Entry RSI: {data['entry_rsi_mean']:.1f} ± {data['entry_rsi_std']:.1f}")
        
    # Load enriched trades for more detailed analysis
    if os.path.exists('data/trades_enriched.csv'):
        enriched_df = pd.read_csv('data/trades_enriched.csv')
        
        print("\n\nDetailed Pattern Analysis from Your Trades:")
        print("="*50)
        
        for trade_type in ['CALL', 'PUT']:
            type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
            if len(type_data) > 0:
                print(f"\n{trade_type} Summary ({len(type_data)} total scenarios):")
                
                # Overall stats
                print(f"  RSI Range: {type_data['Entry_RSI14_W'].min():.1f} - {type_data['Entry_RSI14_W'].max():.1f}")
                print(f"  Win Rate: {(type_data['Exit_Return'] > 0).mean()*100:.0f}%")
                
                # VWAP position analysis
                below_vwap = (type_data['Entry_Price'] < type_data['Entry_VWAP']).sum()
                print(f"  Below VWAP at entry: {below_vwap/len(type_data)*100:.0f}%")
                
                # Volume analysis
                print(f"  Average RVOL: {type_data['Entry_RVOL20'].mean():.2f}x")
                print(f"  Average ATR: {type_data['Entry_ATR14_W'].mean():.3f}")
else:
    print("No trade patterns found. Run trade_analysis_pipeline.py to analyze your trades.")

In [ ]:
# Visualize your actual trading patterns if data exists
if os.path.exists('data/trades_enriched.csv'):
    enriched_df = pd.read_csv('data/trades_enriched.csv')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: RSI Distribution by Trade Type
    for trade_type, color in [('CALL', 'green'), ('PUT', 'red')]:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            axes[0, 0].hist(type_data['Entry_RSI14_W'], bins=15, alpha=0.6, 
                          label=f'{trade_type} (n={len(type_data)})', color=color, edgecolor='black')
    axes[0, 0].set_xlabel('Entry RSI')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Your RSI Distribution at Entry')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Returns by Exit Type
    exit_returns = []
    exit_labels = []
    for trade_type in ['CALL', 'PUT']:
        for exit_type in ['EXIT', 'STOP_LOSS', 'RUNNER']:
            data = enriched_df[(enriched_df['Trade_Type'] == trade_type) & 
                             (enriched_df['Exit_Type'] == exit_type)]
            if len(data) > 0:
                exit_returns.append(data['Exit_Return'].values)
                exit_labels.append(f'{trade_type}\n{exit_type}')
    
    if exit_returns:
        bp = axes[0, 1].boxplot(exit_returns, labels=exit_labels, patch_artist=True)
        # Color the boxes
        for i, patch in enumerate(bp['boxes']):
            if 'CALL' in exit_labels[i]:
                patch.set_facecolor('lightgreen')
            else:
                patch.set_facecolor('lightcoral')
        axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
        axes[0, 1].set_ylabel('Return (%)')
        axes[0, 1].set_title('Your Returns by Trade Type and Exit')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Duration vs Return Scatter
    for trade_type, color in [('CALL', 'green'), ('PUT', 'red')]:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            axes[1, 0].scatter(type_data['Exit_Duration'], type_data['Exit_Return'],
                             alpha=0.6, label=trade_type, color=color, s=50)
    axes[1, 0].set_xlabel('Hold Duration (minutes)')
    axes[1, 0].set_ylabel('Return (%)')
    axes[1, 0].set_title('Your Hold Duration vs Returns')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Plot 4: VWAP Position Analysis
    vwap_data = []
    vwap_labels = []
    for trade_type in ['CALL', 'PUT']:
        type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
        if len(type_data) > 0:
            below = (type_data['Entry_Price'] < type_data['Entry_VWAP']).sum()
            above = len(type_data) - below
            vwap_data.append([below, above])
            vwap_labels.append(trade_type)
    
    if vwap_data:
        x = np.arange(len(vwap_labels))
        width = 0.35
        
        below_counts = [d[0] for d in vwap_data]
        above_counts = [d[1] for d in vwap_data]
        
        axes[1, 1].bar(x - width/2, below_counts, width, label='Below VWAP', color='blue', alpha=0.7)
        axes[1, 1].bar(x + width/2, above_counts, width, label='Above VWAP', color='orange', alpha=0.7)
        
        axes[1, 1].set_ylabel('Number of Trades')
        axes[1, 1].set_title('Your Entry Position Relative to VWAP')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(vwap_labels)
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('Analysis of Your Actual Trading Patterns', y=1.02, fontsize=16)
    plt.show()
else:
    print("No enriched trade data found for visualization.")

## Step 6: Analyze Indicator Values at Entry/Exit

In [ ]:
# Analyze RSI levels at entry for different trade types
print("RSI Analysis at Entry:")
print("="*40)

for trade_type in ['call', 'put']:
    type_df = signals_df[signals_df['trade_type'] == trade_type]
    print(f"\n{trade_type.capitalize()} entries:")
    print(f"Average RSI: {type_df['entry_RSI14_W'].mean():.2f}")
    print(f"Median RSI: {type_df['entry_RSI14_W'].median():.2f}")
    print(f"RSI < 30: {(type_df['entry_RSI14_W'] < 30).sum()} signals")
    print(f"RSI > 70: {(type_df['entry_RSI14_W'] > 70).sum()} signals")

In [ ]:
# Create indicator comparison at entry
indicators_to_compare = ['entry_RSI14_W', 'entry_RVOL20', 'entry_ATR14_W']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, indicator in enumerate(indicators_to_compare):
    for trade_type, color in [('call', 'green'), ('put', 'red')]:
        type_df = signals_df[signals_df['trade_type'] == trade_type]
        axes[idx].hist(type_df[indicator].dropna(), bins=20, alpha=0.5, 
                      label=trade_type, color=color, edgecolor='black')
    
    axes[idx].set_xlabel(indicator.replace('entry_', ''))
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{indicator.replace("entry_", "")} at Entry')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## Step 7: Save Enhanced Data

In [ ]:
# Save the complete dataset with indicators
enhanced_file = output_file.replace('.csv', '_with_indicators.csv')
df.to_csv(enhanced_file, index=False)
print(f"Enhanced data saved to: {enhanced_file}")

# Create a summary report
summary = {
    'Data Range': f"{df['Time'].min()} to {df['Time'].max()}",
    'Total Records': len(df),
    'Total Signals': len(signals_df),
    'Call Signals': len(signals_df[signals_df['trade_type'] == 'call']),
    'Put Signals': len(signals_df[signals_df['trade_type'] == 'put']),
    'Average Signal Return': f"{signals_df['return_pct'].mean():.3f}%",
    'Win Rate': f"{(signals_df['return_pct'] > 0).mean()*100:.1f}%",
    'Files Created': [
        output_file,
        enhanced_file,
        signals_file
    ]
}

print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}: {value}")